# HDB Resale Flat Prices - Python ETL Pipeline

Executable notebook for the technical test, covering data ingestion, validation, deduplication, anomaly detection, transformation, hashing, and reconciliation.

**Processing window:** January 2012 to December 2016  
**Lease calculation date:** 18 July 2026  
**Interpretation:** potential price anomalies are identified and retained for review; they are not treated as invalid records.

In [2]:
from datetime import date
from pathlib import Path
import sys
import tempfile

import pandas as pd
from IPython.display import display


# Support launching Jupyter either from the project root or notebooks/ directory.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from hdb_pipeline.config import PipelineConfig
from hdb_pipeline.ingestion import (
    extract_zip,
    load_master,
    select_source_files,
)
from hdb_pipeline.data_quality import (
    deduplicate,
    flag_price_anomalies,
    validate_and_standardize,
)
from hdb_pipeline.transformation import (
    build_resale_identifier,
    hash_identifiers,
)
from hdb_pipeline.pipeline import run_pipeline


INPUT_ZIP = ROOT / "data" / "input" / "ResaleFlatPrices.zip"
OUTPUT_DIR = ROOT / "output"

CONFIG = PipelineConfig(
    as_of_date=date(2026, 7, 18),
)

if not INPUT_ZIP.exists():
    raise FileNotFoundError(
        f"Input ZIP file was not found: {INPUT_ZIP}"
    )

print(f"Project root: {ROOT}")
print(f"Input file: {INPUT_ZIP}")
print(f"Output directory: {OUTPUT_DIR}")

CONFIG

Project root: /Users/B1/ghome/github/HDB_SDE_Technical_Test_26
Input file: /Users/B1/ghome/github/HDB_SDE_Technical_Test_26/data/input/ResaleFlatPrices.zip
Output directory: /Users/B1/ghome/github/HDB_SDE_Technical_Test_26/output


PipelineConfig(start_month='2012-01', end_month='2016-12', lease_years=99, minimum_category_frequency=2, anomaly_iqr_multiplier=1.5, as_of_date=datetime.date(2026, 7, 18))

## 1. Source discovery and schema comparison

The source ZIP is processed directly. Files within the required date range are selected automatically. Schema union is used because remaining_lease is only available in later files.

In [4]:
with tempfile.TemporaryDirectory() as temporary_directory:
    all_csv_files = extract_zip(
        INPUT_ZIP,
        Path(temporary_directory),
    )

    selected_files = select_source_files(
        all_csv_files,
        CONFIG.start_month,
        CONFIG.end_month,
    )

    schemas = []

    for file_path in selected_files:
        sample = pd.read_csv(file_path, nrows=5)

        schemas.append(
            {
                "file": file_path.name,
                "column_count": len(sample.columns),
                "columns": ", ".join(sample.columns),
            }
        )

    display(pd.DataFrame(schemas))

    master = load_master(selected_files, CONFIG)

print(f"Master rows: {len(master):,}")
display(master.head())

,file,column_count,columns
0,"Resale Flat Prices (Based on Approval Date), 2...",10,"month, town, flat_type, block, street_name, st..."
1,Resale Flat Prices (Based on Registration Date...,11,"month, town, flat_type, block, street_name, st..."
2,Resale Flat Prices (Based on Registration Date...,10,"month, town, flat_type, block, street_name, st..."


Master rows: 92,544


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,source_file,source_row_number
0,2012-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,01 TO 03,44.0,Improved,1979,<NA>,257800.0,"Resale Flat Prices (Based on Approval Date), 2...",366465
1,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,07 TO 09,44.0,Improved,1978,<NA>,263000.0,"Resale Flat Prices (Based on Approval Date), 2...",366466
2,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,10 TO 12,44.0,Improved,1978,<NA>,275000.0,"Resale Flat Prices (Based on Approval Date), 2...",366467
3,2012-01,ANG MO KIO,2 ROOM,170,ANG MO KIO AVE 4,01 TO 03,45.0,Improved,1986,<NA>,260000.0,"Resale Flat Prices (Based on Approval Date), 2...",366468
4,2012-01,ANG MO KIO,2 ROOM,174,ANG MO KIO AVE 4,07 TO 09,45.0,Improved,1986,<NA>,226000.0,"Resale Flat Prices (Based on Approval Date), 2...",366469


## 2. Data profiling

Profiling is performed before rejection and includes null percentages, distinct counts, and categorical frequencies.

In [6]:
profile = pd.DataFrame(
    {
        "column": master.columns,
        "dtype": [str(master[column].dtype) for column in master.columns],
        "null_count": [
            int(master[column].isna().sum())
            for column in master.columns
        ],
        "null_pct": [
            round(master[column].isna().mean() * 100, 4)
            for column in master.columns
        ],
        "distinct_count": [
            int(master[column].nunique(dropna=True))
            for column in master.columns
        ],
    }
)

display(profile)

for column in ["town", "flat_type", "flat_model", "storey_range"]:
    print(f"\n{column} — least frequent values")

    category_counts = (
        master[column]
        .astype("string")
        .str.strip()
        .value_counts(dropna=False)
        .tail(10)
        .rename("row_count")
        .to_frame()
    )

    display(category_counts)

,column,dtype,null_count,null_pct,distinct_count
0,month,string,0,0.0000,60
1,town,object,0,0.0000,26
2,flat_type,object,0,0.0000,7
3,block,string,0,0.0000,2139
4,street_name,object,0,0.0000,522
5,storey_range,object,0,0.0000,25
6,floor_area_sqm,float64,0,0.0000,168
7,flat_model,object,0,0.0000,20
8,lease_commence_date,int64,0,0.0000,48
9,remaining_lease,object,55391,59.8537,50



town — least frequent values


,row_count
town,
GEYLANG,2649
QUEENSTOWN,2524
CLEMENTI,2319
SEMBAWANG,2314
JURONG EAST,2157
SERANGOON,2029
BISHAN,1671
CENTRAL AREA,750
MARINE PARADE,640



flat_type — least frequent values


,row_count
flat_type,
4 ROOM,36535
3 ROOM,26307
5 ROOM,21368
EXECUTIVE,7295
2 ROOM,956
1 ROOM,56
MULTI-GENERATION,27



flat_model — least frequent values


,row_count
flat_model,
Model A-Maisonette,151
Adjoined flat,141
Type S1,138
Terrace,60
Type S2,55
Multi Generation,27
Improved-Maisonette,10
Premium Maisonette,6
Premium Apartment Loft,5



storey_range — least frequent values


,row_count
storey_range,
34 TO 36,88
37 TO 39,81
31 TO 33,79
26 TO 30,39
40 TO 42,38
43 TO 45,8
46 TO 48,8
36 TO 40,7
49 TO 51,2


## 3. Validation strategy

- `month` must follow `YYYY-MM` and be within the configured processing window.
- `town`, `flat_type`, and `flat_model` must meet `CONFIG.minimum_category_frequency`.
- `storey_range` must follow the `NN TO NN` format and its lower value must not exceed its upper value.
- `block` must contain at least one digit, and `street_name` must not be empty.
- `floor_area_sqm`, `lease_commence_date`, and `resale_price` must contain valid positive values.

Rows that fail one or more rules are written to `validation_failed` with machine-readable failure codes.

In [7]:
valid, validation_failed = validate_and_standardize(
    master,
    CONFIG,
)

print(f"Validation passed: {len(valid):,}")
print(f"Validation failed: {len(validation_failed):,}")

display(
    validation_failed[
        [
            "month",
            "town",
            "flat_type",
            "flat_model",
            "failure_reason",
        ]
    ]
)

Validation passed: 92,543
Validation failed: 1


,month,town,flat_type,flat_model,failure_reason
0,2016-10,PASIR RIS,2 ROOM,2-room,INVALID_FLAT_MODEL


## 4. Composite-key duplicate resolution

The key is every original source column except `resale_price`, including original `remaining_lease`. The highest price is retained and lower-priced duplicates are routed to Failed.

In [8]:
deduplicated, duplicate_failed = deduplicate(valid)

# Combine validation failures and duplicate failures into one Failed dataset.
failed = pd.concat(
    [validation_failed, duplicate_failed],
    ignore_index=True,
)

print(f"Rows retained after deduplication: {len(deduplicated):,}")
print(f"Validation failures: {len(validation_failed):,}")
print(f"Duplicate failures: {len(duplicate_failed):,}")
print(f"Total failed rows: {len(failed):,}")

display(duplicate_failed.head(10))
display(
    failed["failure_reason"]
    .value_counts(dropna=False)
    .rename("row_count")
    .to_frame()
)

Rows retained after deduplication: 90,948
Validation failures: 1
Duplicate failures: 1,595
Total failed rows: 1,596


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,source_file,source_row_number,month_date,remaining_lease_years,remaining_lease_months,remaining_lease_recomputed,remaining_lease_as_of_date,failure_reason
0,2012-01,ANG MO KIO,3 ROOM,256,ANG MO KIO AVE 4,10 TO 12,73.0,New Generation,1977,<NA>,340000.0,"Resale Flat Prices (Based on Approval Date), 2...",366496,2012-01-01,49,6,49 years 06 months,2026-07-18,DUPLICATE_LOWER_PRICE
1,2012-01,ANG MO KIO,3 ROOM,506,ANG MO KIO AVE 8,07 TO 09,68.0,New Generation,1980,<NA>,334000.0,"Resale Flat Prices (Based on Approval Date), 2...",366510,2012-01-01,52,6,52 years 06 months,2026-07-18,DUPLICATE_LOWER_PRICE
2,2012-01,ANG MO KIO,3 ROOM,558,ANG MO KIO AVE 10,10 TO 12,67.0,New Generation,1980,<NA>,336000.0,"Resale Flat Prices (Based on Approval Date), 2...",366480,2012-01-01,52,6,52 years 06 months,2026-07-18,DUPLICATE_LOWER_PRICE
3,2012-01,ANG MO KIO,3 ROOM,633,ANG MO KIO AVE 6,10 TO 12,67.0,New Generation,1985,<NA>,350000.0,"Resale Flat Prices (Based on Approval Date), 2...",366508,2012-01-01,57,6,57 years 06 months,2026-07-18,DUPLICATE_LOWER_PRICE
4,2012-01,BEDOK,3 ROOM,138,BEDOK NTH ST 2,10 TO 12,67.0,New Generation,1978,<NA>,316000.0,"Resale Flat Prices (Based on Approval Date), 2...",366574,2012-01-01,50,6,50 years 06 months,2026-07-18,DUPLICATE_LOWER_PRICE
5,2012-01,BUKIT MERAH,3 ROOM,37,JLN RUMAH TINGGI,04 TO 06,53.0,Standard,1969,<NA>,303000.0,"Resale Flat Prices (Based on Approval Date), 2...",366750,2012-01-01,41,6,41 years 06 months,2026-07-18,DUPLICATE_LOWER_PRICE
6,2012-01,BUKIT MERAH,3 ROOM,37,JLN RUMAH TINGGI,04 TO 06,53.0,Standard,1969,<NA>,290000.0,"Resale Flat Prices (Based on Approval Date), 2...",366753,2012-01-01,41,6,41 years 06 months,2026-07-18,DUPLICATE_LOWER_PRICE
7,2012-01,CHOA CHU KANG,4 ROOM,485,CHOA CHU KANG AVE 5,07 TO 09,102.0,Model A,1999,<NA>,385000.0,"Resale Flat Prices (Based on Approval Date), 2...",366883,2012-01-01,71,6,71 years 06 months,2026-07-18,DUPLICATE_LOWER_PRICE
8,2012-01,GEYLANG,3 ROOM,23,BALAM RD,07 TO 09,60.3,Standard,1967,<NA>,298000.0,"Resale Flat Prices (Based on Approval Date), 2...",366989,2012-01-01,39,6,39 years 06 months,2026-07-18,DUPLICATE_LOWER_PRICE
9,2012-01,HOUGANG,3 ROOM,682,HOUGANG AVE 4,04 TO 06,64.0,Simplified,1989,<NA>,319000.0,"Resale Flat Prices (Based on Approval Date), 2...",367038,2012-01-01,61,6,61 years 06 months,2026-07-18,DUPLICATE_LOWER_PRICE


,row_count
failure_reason,
DUPLICATE_LOWER_PRICE,1595
INVALID_FLAT_MODEL,1


## 5. Remaining lease and anomaly review

**Remaining lease**

* Assume a **99-year lease** starting on **1 January** of the `lease_commence_date` year.
* Calculate the remaining lease as of the transaction month.
* Express the result in **completed years and months**, rounding down partial months.

**Price anomaly detection**

* Calculate `price_per_sqm` for each transaction.
* Group records by **year**, **town**, and **flat_type**.
* Within each group:

  * Calculate the first quartile (**Q1**) and third quartile (**Q3**).
  * Compute the interquartile range (**IQR = Q3 − Q1**).
  * Flag records where:

    * `price_per_sqm < Q1 − 1.5 × IQR`, or
    * `price_per_sqm > Q3 + 1.5 × IQR`.
* Flagged records are **retained in the Cleaned dataset** and **copied to the Review dataset** for manual investigation.

In [10]:
cleaned_preview, anomaly_review = flag_price_anomalies(
    deduplicated,
    CONFIG,
)

print(f"Potential anomalies flagged: {len(anomaly_review):,}")

display(
    cleaned_preview[
        [
            "lease_commence_date",
            "remaining_lease_years",
            "remaining_lease_months",
            "remaining_lease_recomputed",
        ]
    ].head()
)

display(
    anomaly_review[
        [
            "month",
            "town",
            "flat_type",
            "floor_area_sqm",
            "resale_price",
            "price_per_sqm",
            "is_anomalous_price",
            "anomaly_reason",
        ]
    ].head(10)
)

Potential anomalies flagged: 2,010


,lease_commence_date,remaining_lease_years,remaining_lease_months,remaining_lease_recomputed
0,1979,51,6,51 years 06 months
1,1978,50,6,50 years 06 months
2,1978,50,6,50 years 06 months
3,1986,58,6,58 years 06 months
4,1986,58,6,58 years 06 months


,month,town,flat_type,floor_area_sqm,resale_price,price_per_sqm,is_anomalous_price,anomaly_reason
0,2012-01,ANG MO KIO,2 ROOM,45.0,226000.0,5022.222222,True,POTENTIAL_PRICE_ANOMALY_IQR
1,2012-01,BUKIT MERAH,3 ROOM,60.0,485000.0,8083.333333,True,POTENTIAL_PRICE_ANOMALY_IQR
2,2012-01,BUKIT PANJANG,3 ROOM,73.0,301000.0,4123.287671,True,POTENTIAL_PRICE_ANOMALY_IQR
3,2012-01,GEYLANG,3 ROOM,60.0,418000.0,6966.666667,True,POTENTIAL_PRICE_ANOMALY_IQR
4,2012-01,HOUGANG,3 ROOM,80.0,325000.0,4062.500000,True,POTENTIAL_PRICE_ANOMALY_IQR
5,2012-01,HOUGANG,3 ROOM,82.0,340000.0,4146.341463,True,POTENTIAL_PRICE_ANOMALY_IQR
6,2012-01,HOUGANG,4 ROOM,100.0,334000.0,3340.000000,True,POTENTIAL_PRICE_ANOMALY_IQR
7,2012-01,HOUGANG,5 ROOM,125.0,442888.0,3543.104000,True,POTENTIAL_PRICE_ANOMALY_IQR
8,2012-01,HOUGANG,EXECUTIVE,152.0,475000.0,3125.000000,True,POTENTIAL_PRICE_ANOMALY_IQR
9,2012-01,JURONG WEST,3 ROOM,67.0,246000.0,3671.641791,True,POTENTIAL_PRICE_ANOMALY_IQR


## 6. Resale Identifier and SHA-256

The prescribed `Resale Identifier` is deterministic but may not be unique. Collisions are reported, not removed.

SHA-256 hashes the Identifier without changing the number of distinct values.


In [11]:
transformed_preview = build_resale_identifier(
    cleaned_preview
)
hashed_preview = hash_identifiers(
    transformed_preview
)

print(f"Rows: {len(transformed_preview):,}")
print(
    "Distinct identifiers: "
    f"{transformed_preview['resale_identifier'].nunique():,}"
)
print(
    "Distinct hashes: "
    f"{hashed_preview['hashed_resale_identifier'].nunique():,}"
)
print(
    "Rows participating in identifier collisions: "
    f"{transformed_preview.duplicated('resale_identifier', keep=False).sum():,}"
)

display(
    transformed_preview[
        [
            "month",
            "town",
            "flat_type",
            "block",
            "resale_price",
            "resale_identifier",
        ]
    ].head()
)

Rows: 90,948
Distinct identifiers: 77,254
Distinct hashes: 77,254
Rows participating in identifier collisions: 24,645


,month,town,flat_type,block,resale_price,resale_identifier
0,2012-01,ANG MO KIO,2 ROOM,406,257800.0,S4062501A
1,2012-01,ANG MO KIO,2 ROOM,314,263000.0,S3142501A
2,2012-01,ANG MO KIO,2 ROOM,314,275000.0,S3142501A
3,2012-01,ANG MO KIO,2 ROOM,170,260000.0,S1702501A
4,2012-01,ANG MO KIO,2 ROOM,174,226000.0,S1742501A


## 7. Execute the complete pipeline and reconcile outputs

In [12]:
manifest = run_pipeline(
    INPUT_ZIP,
    OUTPUT_DIR,
    CONFIG,
)

display(
    pd.DataFrame([manifest["row_counts"]])
    .T
    .rename(columns={0: "rows"})
)

display(
    pd.DataFrame([manifest["quality_checks"]])
    .T
    .rename(columns={0: "passed"})
)

counts = manifest["row_counts"]

assert counts["master"] == counts["cleaned"] + counts["failed_total"]
assert counts["cleaned"] == counts["transformed"] == counts["hashed"]
assert manifest["quality_checks"]["reconciled"]
assert manifest["quality_checks"]["anomalies_retained_in_cleaned"]
assert manifest["quality_checks"]["hash_cardinality_preserved"]
assert manifest["quality_checks"]["cleaned_has_no_duplicate_keys"]

print("All reconciliation and quality assertions passed.")

,rows
master,92544
validation_failed,1
duplicate_failed,1595
anomaly_review,2010
cleaned,90948
transformed,90948
hashed,90948
failed_total,1596


,passed
reconciled,True
anomalies_retained_in_cleaned,True
hash_cardinality_preserved,True
cleaned_has_no_duplicate_keys,True


All reconciliation and quality assertions passed.


## 8. Output contract

| Group | Content |
|---|---|
| Raw | Unmodified contributing source CSVs |
| Staging | Schema-unioned in-scope master |
| Cleaned | Valid, deduplicated data with anomaly flags |
| Transformed | Cleaned plus clear-text Resale Identifier |
| Failed | Invalid rows and lower-price composite-key duplicates |
| Review | Potential price anomalies, also retained in Cleaned |
| Hashed | Cleaned plus SHA-256 hashed Identifier |
| Profiling | Column profiles and category distributions |